# 02_02_Profiling and Cleaning `municipalities` Dataset #

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
path_raw = Path("../data/raw/")

path_intermediate = Path("../data/intermediate/")

# Table of Contents

- [2. MUNICIPALITY DATASET](#2-municipality-dataset)
- [2.1. Data Profiling](#21-data-profiling)
    - [2.1.1. Overview](#211-overview)
    - [2.1.2. Profiling the hierarchy of geographic entities](#212-profiling-the-administrative-entities-hierarchy)
- [2.2. Data Quality and Cleaning](#22-data-quality--cleaning)
    - [2.2.1. Handling Duplicates](#221-handling-duplicates)
    - [2.2.2. Cleaning String Values](#222-cleaning-string-values)
    - [2.2.3. Removing Outdated Records](#223-removing-outdated-records)
    - [2.2.4. Dropping Irrelevant Columns](#224-dropping-irrelevant-columns)
    - [2.2.5. Hierarchy Flattening](#225-hierarchy-flattening)
- [2.3. Export to Silver Layer](#23-export-to-silver-layer)

# 2. MUNICIPALITY DATASET #

## 2.1. Data Profiling ##

### 2.1.1. Overview ###

As shown below:  

* None of the columns are unique. However, with 2773 distinct values out of 2781 entries, the `CD_REFNIS` is the best candidate as a **Business Key**. 

* There are **4 hierarchy levels** in the dataset.  

* The `CD_SUP_REFNIS` column is of type `str` instead of `int`, as it contains a `"-"` for Level 1 entities (Regions), which have no parent level.

* There are **no missing values**.

* `CD_REFNIS` countains **9 duplicate values**.

In [3]:
municipalities = pd.read_excel(path_raw / "communes.xlsx")
municipalities.head()

,LVL_REFNIS,CD_REFNIS,CD_SUP_REFNIS,TX_REFNIS_DE,TX_REFNIS_FR,TX_REFNIS_NL,DT_VLDT_START,DT_VLDT_END
0,1,2000,-,Flämische Region,Région flamande,Vlaams Gewest,01/01/1970,31/12/9999
1,1,3000,-,Wallonische Region,Région wallonne,Waals Gewest,01/01/1970,31/12/9999
2,1,4000,-,Region Brüssel-Hauptstadt,Région de Bruxelles-Capitale,Brussels Hoofdstedelijk Gewest,01/01/1970,31/12/9999
3,2,10000,02000,Provinz Antwerpen,Province d’Anvers,Provincie Antwerpen,01/01/1970,31/12/9999
4,2,20000,-,Provinz Brabant,Province de Brabant,Provincie Brabant,01/01/1970,31/12/1994


In [4]:
municipalities["TX_REFNIS_FR"].head(20)

0                          Région flamande
1                          Région wallonne
2             Région de Bruxelles-Capitale
3                        Province d’Anvers
4                      Province de Brabant
5              Province du Brabant flamand
6               Province du Brabant wallon
7          Province de Flandre occidentale
8            Province de Flandre orientale
9                      Province du Hainaut
10                       Province de Liège
11                    Province du Limbourg
12                  Province du Luxembourg
13                       Province de Namur
14                 Arrondissement d’Anvers
15               Arrondissement de Malines
16              Arrondissement de Turnhout
17    Arrondissement de Bruxelles-Capitale
18          Arrondissement de Hal-Vilvorde
19          Arrondissement de Hal-Vilvorde
Name: TX_REFNIS_FR, dtype: str

In [5]:
municipalities.info()

<class 'pandas.DataFrame'>
RangeIndex: 2782 entries, 0 to 2781
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   LVL_REFNIS     2782 non-null   int64 
 1   CD_REFNIS      2782 non-null   int64 
 2   CD_SUP_REFNIS  2782 non-null   str   
 3   TX_REFNIS_DE   2782 non-null   str   
 4   TX_REFNIS_FR   2782 non-null   str   
 5   TX_REFNIS_NL   2782 non-null   str   
 6   DT_VLDT_START  2782 non-null   object
 7   DT_VLDT_END    2782 non-null   object
dtypes: int64(2), object(2), str(4)
memory usage: 267.4+ KB


In [6]:
municipalities.nunique()

LVL_REFNIS          4
CD_REFNIS        2773
CD_SUP_REFNIS      59
TX_REFNIS_DE     2756
TX_REFNIS_FR     2755
TX_REFNIS_NL     2755
DT_VLDT_START      14
DT_VLDT_END        15
dtype: int64

In [7]:
municipalities["CD_REFNIS"].duplicated().sum()

np.int64(9)

In [8]:
municipalities.isnull().sum()

LVL_REFNIS       0
CD_REFNIS        0
CD_SUP_REFNIS    0
TX_REFNIS_DE     0
TX_REFNIS_FR     0
TX_REFNIS_NL     0
DT_VLDT_START    0
DT_VLDT_END      0
dtype: int64

In [9]:
municipalities.columns

Index(['LVL_REFNIS', 'CD_REFNIS', 'CD_SUP_REFNIS', 'TX_REFNIS_DE',
       'TX_REFNIS_FR', 'TX_REFNIS_NL', 'DT_VLDT_START', 'DT_VLDT_END'],
      dtype='str')

In [10]:
municipalities.duplicated().sum()

np.int64(0)

### 2.1.2. Profiling the hierarchy of geographic entities ###

As shown below, the hierarchy of Belgian geographic entities consists of 4 levels:  
* **Level 1** : Regions  
* **Level 2** : Provinces  
* **Level 3** : Arrondissements  
* **Level 4** : Municipalities

Except for the level 1 (the regions), each entity is linked to its parent level via the **parent REFNIS code**, stored in the `CD_SUP_REFNIS` column.  

In section 2.2.4. of this notebook, we will **flatten this hierarchy** in order to build a **denormalized dataset** that will enrich the future `station` dimension, once merged with the `station_cleaned` dataset.  

Since the level 3 (the arrondissements) is irrelevant for our analysis, it will be excluded from this dataset during the flattening process.

In [11]:
municipalities.groupby("LVL_REFNIS")[["TX_REFNIS_FR", "CD_SUP_REFNIS"]].agg(["count", "first"])

TX_REFNIS_FR                          CD_SUP_REFNIS       
                  count                    first         count  first
LVL_REFNIS                                                           
1                     3          Région flamande             3      -
2                    11        Province d’Anvers            11  02000
3                    48  Arrondissement d’Anvers            48  10000
4                  2720               Aartselaar          2720  11000

## 2.2. Data Quality & Cleaning ##

### 2.2.1. Handling Duplicates ###

The following two outputs show that the **9 duplicate REFNIS codes** are historical REFNIS codes from administrative entites that have since been renamed (including spelling standardisation), merged, dissolved, or reassigned a new code.

**Duplicates caused by name or spelling changes**  

57000 | "Arrondissement de Tournai-Mouscron" -> "Arrondissement de Tournai" | Valid from 1970
52048 | "Montignies-le-Tilleul" -> "Montigny-le-Tilleul"                    | Valid from 1976
55035 | "Le Roeulx" -> "Roeulx"                                             | Valid from 1971
56029 | "Froid-chapelle" -> "Froidchapelle"                                 | Valid from 1976
92137 | "Basse-Sambre" -> "Sambreville"                                     | Valid from 1979  

**Duplicates caused by outdated REFNIS codes**  

23000 | "Arrondissement de Hal-Vilvorde" | 01/01/1970 -> 01/01/1995  
24000 | "Arrondissement Leuven"          | 01/01/1970 -> 01/01/1995  
25000 | "Arrondissement de Nivelles"     | 01/01/1970 -> 01/01/1995  
61013 | "Clermont (Huy)"                 | 01/01/1970 -> 01/01/1972  

**Decisions**:
* Duplicated related to arrondissements are excluded, as this administrative level is not relevant to our analysis.  
* For names and spelling changes, only the most recent record is retained to ensure that the analysis is based on the current official names.  
* After removing arrondissements and outdated records, only one duplicate remains: **REFNIS code 61013 - "Clermont (Huy)"**. This case is investigated in more detail below.

In [12]:
duplicated_refnis = (
        municipalities[municipalities["CD_REFNIS"].duplicated()]
        [["CD_REFNIS", "TX_REFNIS_FR", "DT_VLDT_END"]]
    )

print(duplicated_refnis)

      CD_REFNIS                        TX_REFNIS_FR DT_VLDT_END
19        23000      Arrondissement de Hal-Vilvorde  31/12/1994
21        24000           Arrondissement de Louvain  31/12/1994
23        25000          Arrondissement de Nivelles  31/12/1994
45        57000  Arrondissement de Tournai-Mouscron  31/12/9999
1251      52048               Montignies-le-Tilleul  31/12/1975
1407      55035                              Roeulx  31/12/1970
1452      56029                      Froid-chapelle  31/12/1975
1626      61013                      Clermont (Huy)  31/12/1971
2686      92137                        Basse-Sambre  31/12/1978


In [13]:
duplicated_rows = municipalities[municipalities["CD_REFNIS"].duplicated(keep=False)]

display(duplicated_rows)

,LVL_REFNIS,CD_REFNIS,CD_SUP_REFNIS,TX_REFNIS_DE,TX_REFNIS_FR,TX_REFNIS_NL,DT_VLDT_START,DT_VLDT_END
18,3,23000,20001,Bezirk Halle-Vilvoorde,Arrondissement de Hal-Vilvorde,Arrondissement Halle-Vilvoorde,01/01/1995,31/12/9999
19,3,23000,20000,Bezirk Halle-Vilvoorde,Arrondissement de Hal-Vilvorde,Arrondissement Halle-Vilvoorde,01/01/1970,31/12/1994
20,3,24000,20001,Bezirk Löwen,Arrondissement de Louvain,Arrondissement Leuven,01/01/1995,31/12/9999
21,3,24000,20000,Bezirk Löwen,Arrondissement de Louvain,Arrondissement Leuven,01/01/1970,31/12/1994
22,3,25000,20002,Bezirk Nivelles,Arrondissement de Nivelles,Arrondissement Nijvel,01/01/1995,31/12/9999
23,3,25000,20000,Bezirk Nivelles,Arrondissement de Nivelles,Arrondissement Nijvel,01/01/1970,31/12/1994
44,3,57000,50000,Bezirk Tournai,Arrondissement de Tournai,Arrondissement Doornik,01/01/1970,2018-12-31 00:00:00
45,3,57000,50000,Bezirk Tournai-Mouscron,Arrondissement de Tournai-Mouscron,Arrondissement Doornik-Moeskroen,01/01/2019,31/12/9999
1250,4,52048,52000,Montigny-le-Tilleul,Montigny-le-Tilleul,Montigny-le-Tilleul,01/01/1976,31/12/9999
1251,4,52048,52000,Montignies-le-Tilleul,Montignies-le-Tilleul,Montignies-le-Tilleul,01/01/1970,31/12/1975


Investigation reveals that **Clermont (Huy)** ceased to be a Belgian municipality following the *Fusion of the Belgian Municipalities* in 1977. It is now part of the **Engis municipality** (REFNIS codes : 61080 and 62031).  

However, since it retains its own REFNIS code, we do not exclude it from the dataset. As decided above, we only keep the most recent entry.

In [14]:
municipalities["DT_VLDT_START"] = pd.to_datetime(municipalities["DT_VLDT_START"], format='%d/%m/%Y')

In [15]:
dates_to_keep = municipalities.groupby("CD_REFNIS")["DT_VLDT_START"].idxmax()

municipalities = municipalities.loc[dates_to_keep].reset_index(drop=True)

In [16]:
municipalities["CD_REFNIS"].duplicated().sum()

np.int64(0)

The `CD_REFNIS` column now contains **no duplicate values**.

### 2.2.2. Cleaning String Values ###

String values in `TX_REFNIS_FR` and `TX_REFNIS_NL` are preserved in their original format, as they are intended for **display purposes** only and will never be used as join keys. Only leading and trailing whitespace is removed.

In [17]:
for col in ["TX_REFNIS_FR", "TX_REFNIS_NL"]:
    municipalities[col] = municipalities[col].str.strip()

### 2.2.3. Removing Outdated Records

For our analysis, we only retain records of entities that are still active.

In [18]:
municipalities = municipalities[municipalities["DT_VLDT_END"] == "31/12/9999"].reset_index(drop=True)

### 2.2.4. Dropping Irrelevant Columns ###

* Since we already have the names of the administrative entities in Dutch and French, we **exclude the german version**: the `TX_REFNIS_DE` column. 

* Now that we have handled the duplicate values and the outdated records, we can also **drop the date-related columns** `DT_VLDT_START` and `DT_VLDT_END`.  

In [19]:
columns_to_drop = ["TX_REFNIS_DE", "DT_VLDT_START", "DT_VLDT_END"]

municipalities = municipalities.drop(columns=columns_to_drop)

In [20]:
municipalities.columns

Index(['LVL_REFNIS', 'CD_REFNIS', 'CD_SUP_REFNIS', 'TX_REFNIS_FR',
       'TX_REFNIS_NL'],
      dtype='str')

### 2.2.5. Hierarchy Flattening ###

To build the future `station` dimension of our data warehouse and **to map the operational points of the Infrabel network on a Power BI dashboard**, the station_cleaned dataset will need to be enriched with the data contained in the municipalities dataframe - placing each operational point in its corresponding municipality. To achieve this, the hierarchy contained in this dataframe must first be **flattened**.

To flatten the hierarchy in `municipalities`: 
* We create **four dataframes** - one per hierarchy level.
* Then, we perform **three successive self-joins** using the `CD_REFNIS` as the primary key, and the `CD_SUP_REFNIS` columns as the foreign key.
* Finally, we **drop the irrelevant columns**.

In the process, we also:
* convert the `CD_SUP_REFNIS` column from `str` to `int`, 
* handle the 19 Brussels municipalities, which have no *Province level*,
* rename columns for display in future dashboards, 
* and drop the intermediate columns created by the successive self-joins.

First, we replace the "-" character in the `CD_SUP_REFNIS` column with `0`, before converting it from `str` to `int`.

In [21]:
municipalities["CD_SUP_REFNIS"] = municipalities["CD_SUP_REFNIS"].replace("-", "0")

municipalities["CD_SUP_REFNIS"] = municipalities["CD_SUP_REFNIS"].astype(int)

Second, we create the four dataframes, rename the `TX_REFNIS_FR` and the `TX_REFNIS_NL`columns in each dataframe, and drop the intermediate or irrelevant columns.

In [22]:
regions = municipalities.loc[municipalities["LVL_REFNIS"] == 1]

regions = regions.rename(columns={"TX_REFNIS_FR" : "Régions", "TX_REFNIS_NL" : "Gewesten"})

In [23]:
regions_columns_to_drop = ["CD_SUP_REFNIS", "LVL_REFNIS"]

regions = regions.drop(columns=regions_columns_to_drop)
regions.head()

,CD_REFNIS,Régions,Gewesten
0,2000,Région flamande,Vlaams Gewest
1,3000,Région wallonne,Waals Gewest
2,4000,Région de Bruxelles-Capitale,Brussels Hoofdstedelijk Gewest


In [24]:
provinces = municipalities.loc[municipalities["LVL_REFNIS"] == 2]

provinces = provinces.drop(columns="LVL_REFNIS")

provinces = provinces.rename(columns={"TX_REFNIS_FR" : "Provinces", "TX_REFNIS_NL" : "Provincies"})
provinces.head(10)


,CD_REFNIS,CD_SUP_REFNIS,Provinces,Provincies
3,10000,2000,Province d’Anvers,Provincie Antwerpen
73,20001,2000,Province du Brabant flamand,Provincie Vlaams-Brabant
74,20002,3000,Province du Brabant wallon,Provincie Waals-Brabant
188,30000,2000,Province de Flandre occidentale,Provincie West-Vlaanderen
259,40000,2000,Province de Flandre orientale,Provincie Oost-Vlaanderen
321,50000,3000,Province du Hainaut,Provincie Henegouwen
398,60000,3000,Province de Liège,Provincie Luik
487,70000,2000,Province du Limbourg,Provincie Limburg
529,80000,3000,Province du Luxembourg,Provincie Luxemburg
578,90000,3000,Province de Namur,Provincie Namen


In [25]:
arrondissements = municipalities.loc[municipalities["LVL_REFNIS"] == 3]

arrondissements = arrondissements.drop(columns="LVL_REFNIS")

arrondissements = (arrondissements
                   .rename(columns={"TX_REFNIS_FR" : "Arrondissements", 
                                    "TX_REFNIS_NL" : "Arrondissementen"})
                )
arrondissements.head()

,CD_REFNIS,CD_SUP_REFNIS,Arrondissements,Arrondissementen
4,11000,10000,Arrondissement d’Anvers,Arrondissement Antwerpen
32,12000,10000,Arrondissement de Malines,Arrondissement Mechelen
45,13000,10000,Arrondissement de Turnhout,Arrondissement Turnhout
75,21000,4000,Arrondissement de Bruxelles-Capitale,Arrondissement Brussel-Hoofdstad
95,23000,20001,Arrondissement de Hal-Vilvorde,Arrondissement Halle-Vilvoorde


In [26]:
municipalities_only = municipalities.loc[municipalities["LVL_REFNIS"] == 4]

municipalities_only = municipalities_only.drop(columns="LVL_REFNIS")

municipalities_only = (municipalities_only
                       .rename(columns={"TX_REFNIS_FR" : "Communes", 
                                        "TX_REFNIS_NL" : "Gemeenten"})
                    )
municipalities_only.head()

,CD_REFNIS,CD_SUP_REFNIS,Communes,Gemeenten
5,11001,11000,Aartselaar,Aartselaar
6,11004,11000,Boechout,Boechout
7,11005,11000,Boom,Boom
8,11008,11000,Brasschaat,Brasschaat
9,11009,11000,Brecht,Brecht


Third, we perform the **three successive self-joins** and handle the **Brussels municipalities exception**:  
1. Merging municipalities_only with arrondissements, storing the result as `mun_arr`.
2. Merging `mun_arr` with `provinces`, storing the result as `mun_prov`.
3. Handling the Brussels municipalities exception in `mun_prov`.
4. Merging `mun_prov` with `regions`, storing the final flattened dataframe as `municipalities_flattened`.

In [27]:
mun_arr = pd.merge(
    municipalities_only,
    arrondissements,
    how="left",
    left_on="CD_SUP_REFNIS",
    right_on="CD_REFNIS"
)

mun_arr.columns

Index(['CD_REFNIS_x', 'CD_SUP_REFNIS_x', 'Communes', 'Gemeenten',
       'CD_REFNIS_y', 'CD_SUP_REFNIS_y', 'Arrondissements',
       'Arrondissementen'],
      dtype='str')

In [28]:
mun_arr = mun_arr.rename(columns={
    "CD_REFNIS_x": "CD_REFNIS_mun",
    "CD_REFNIS_y": "CD_REFNIS_arr",
    "CD_SUP_REFNIS_x": "CD_SUP_REFNIS_mun",
    "CD_SUP_REFNIS_y": "CD_SUP_REFNIS_arr"
})

In [29]:
mun_prov = pd.merge(
    mun_arr,
    provinces,
    how="left",
    left_on="CD_SUP_REFNIS_arr",
    right_on="CD_REFNIS"
)

mun_prov.columns

Index(['CD_REFNIS_mun', 'CD_SUP_REFNIS_mun', 'Communes', 'Gemeenten',
       'CD_REFNIS_arr', 'CD_SUP_REFNIS_arr', 'Arrondissements',
       'Arrondissementen', 'CD_REFNIS', 'CD_SUP_REFNIS', 'Provinces',
       'Provincies'],
      dtype='str')

In [30]:
mun_prov = mun_prov.rename(columns={
    "CD_REFNIS": "CD_REFNIS_prov",
    "CD_SUP_REFNIS": "CD_SUP_REFNIS_prov"
})

In [31]:
mun_prov.isnull().sum()

CD_REFNIS_mun          0
CD_SUP_REFNIS_mun      0
Communes               0
Gemeenten              0
CD_REFNIS_arr          0
CD_SUP_REFNIS_arr      0
Arrondissements        0
Arrondissementen       0
CD_REFNIS_prov        19
CD_SUP_REFNIS_prov    19
Provinces             19
Provincies            19
dtype: int64

In [32]:
mun_prov["Provinces"] = mun_prov["Provinces"].fillna("Brussels-Capital Region")
mun_prov["Provincies"] = mun_prov["Provincies"].fillna("Brussels Hoofdstedelijk Gewest")
mun_prov["CD_REFNIS_prov"] = mun_prov["CD_REFNIS_prov"].fillna(4000)
mun_prov["CD_SUP_REFNIS_prov"] = mun_prov["CD_SUP_REFNIS_prov"].fillna(4000)

In [33]:
mun_prov["CD_REFNIS_prov"] = mun_prov["CD_REFNIS_prov"].astype(int)
mun_prov["CD_SUP_REFNIS_prov"] = mun_prov["CD_SUP_REFNIS_prov"].astype(int)

In [34]:
municipalities_flattened = pd.merge(
    mun_prov,
    regions,
    how="left",
    left_on="CD_SUP_REFNIS_prov",
    right_on="CD_REFNIS"
)
municipalities_flattened.columns

Index(['CD_REFNIS_mun', 'CD_SUP_REFNIS_mun', 'Communes', 'Gemeenten',
       'CD_REFNIS_arr', 'CD_SUP_REFNIS_arr', 'Arrondissements',
       'Arrondissementen', 'CD_REFNIS_prov', 'CD_SUP_REFNIS_prov', 'Provinces',
       'Provincies', 'CD_REFNIS', 'Régions', 'Gewesten'],
      dtype='str')

In [35]:
municipalities_flattened = municipalities_flattened.rename(columns={
    "CD_REFNIS": "CD_REFNIS_region"
})

As a final step, we drop the intermediate columns resulting from the successive merging, and the *arrondissements* related columns, as there are irrelevant to our analysis.

In [36]:
mun_flat_col_to_drop = ["CD_SUP_REFNIS_mun",
                        "CD_SUP_REFNIS_arr",
                        "CD_SUP_REFNIS_prov",
                        "Arrondissements",
                        "Arrondissementen",
                        "CD_REFNIS_arr"]

municipalities_flattened = municipalities_flattened.drop(columns=mun_flat_col_to_drop)

In [37]:
municipalities_flattened.head()

,CD_REFNIS_mun,Communes,Gemeenten,CD_REFNIS_prov,Provinces,Provincies,CD_REFNIS_region,Régions,Gewesten
0,11001,Aartselaar,Aartselaar,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
1,11004,Boechout,Boechout,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
2,11005,Boom,Boom,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
3,11008,Brasschaat,Brasschaat,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
4,11009,Brecht,Brecht,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest


In [38]:
municipalities_flattened.nunique()

CD_REFNIS_mun       564
Communes            564
Gemeenten           564
CD_REFNIS_prov       11
Provinces            11
Provincies           11
CD_REFNIS_region      3
Régions               3
Gewesten              3
dtype: int64

In [39]:
municipalities_flattened.duplicated().sum()

np.int64(0)

As shown in the outputs above, the flattened dataframe is now ready for the export to the intermediate directory.

## 2.3. Export to Silver Layer ##

In [40]:
municipalities_flattened.to_parquet(path_intermediate / "municipalities_flattened.parquet")

In [41]:
df_to_verify = pd.read_parquet(path_intermediate / "municipalities_flattened.parquet")
print(df_to_verify.shape, df_to_verify.dtypes.to_dict())
df_to_verify.head()

(564, 9) {'CD_REFNIS_mun': dtype('int64'), 'Communes': <StringDtype(na_value=nan)>, 'Gemeenten': <StringDtype(na_value=nan)>, 'CD_REFNIS_prov': dtype('int64'), 'Provinces': <StringDtype(na_value=nan)>, 'Provincies': <StringDtype(na_value=nan)>, 'CD_REFNIS_region': dtype('int64'), 'Régions': <StringDtype(na_value=nan)>, 'Gewesten': <StringDtype(na_value=nan)>}


,CD_REFNIS_mun,Communes,Gemeenten,CD_REFNIS_prov,Provinces,Provincies,CD_REFNIS_region,Régions,Gewesten
0,11001,Aartselaar,Aartselaar,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
1,11004,Boechout,Boechout,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
2,11005,Boom,Boom,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
3,11008,Brasschaat,Brasschaat,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
4,11009,Brecht,Brecht,10000,Province d’Anvers,Provincie Antwerpen,2000,Région flamande,Vlaams Gewest
